# Figure 4: Intrinsic Spinal Gradient Organization - 33 ROI

This notebook reproduces **Figure 4** from the manuscript. It assumes precomputed timecourses and connectivity matrices are available on disk, as configured via `config-results.json`. Anatomical parcellation used :  198 spinal cord ROIs - Spanning C3–C8 spinal levels - 6 levels with 33 ROI each - PAM50 Atlas, are used for constructing the corticospinal FC matrix.


## Config and imports

This cell loads the configuration file and imports all required libraries and utility functions.

In [ ]:
from utils import *

# Load analysis config
params = read_config('config-results.json')

n_rois = 400
schaefer_dataset = datasets.fetch_atlas_schaefer_2018(n_rois=n_rois, resolution_mm=2)
schaefer_atlas = schaefer_dataset.maps
schaefer_labels = (schaefer_dataset.labels).astype(str)

sc_data = load_img(params["custom_sc_atlas"]).get_fdata()
sc_labels = open(params["custom_sc_labels"],'r').read().splitlines()

## Panel 4A

This cell loads the cortcospinal FC with SMC rois and 33-ROI parcellation of spinal cord. 
Computes mean FC, performs sparsification. 
Visualizes 33-ROI paracellation spinal only FC. 
Computing Gradients, Reference, Alignment, Variance explained, Gradient maps for 33-ROI case

#### Load corticopinal FC, ROI list 

In [ ]:
fc_dir = params["save_fc_mats_33sp"]
fc_files = sorted(glob.glob(os.path.join(fc_dir, "*_subFC_33sp.csv")))

if not fc_files:
    raise FileNotFoundError(
        f"No '*_subFC_33sp.csv' files found in:\n{fc_dir}"
    )

subFC_mats = []
sub_rois = None

for fpath in fc_files:
    df_fc = pd.read_csv(fpath, index_col=0)

    # Enforce a square, consistently labelled FC matrix.
    if df_fc.shape[0] != df_fc.shape[1]:
        raise ValueError(
            f"FC matrix is not square: {os.path.basename(fpath)} "
            f"has shape {df_fc.shape}."
        )

    if df_fc.index.tolist() != df_fc.columns.tolist():
        raise ValueError(
            f"Row/column ROI labels differ in: {os.path.basename(fpath)}"
        )

    # Save the ROI order from the first subject and compare all others to it.
    if sub_rois is None:
        sub_rois = df_fc.index.tolist()
    elif df_fc.index.tolist() != sub_rois:
        raise ValueError(
            f"ROI ordering mismatch in: {os.path.basename(fpath)}"
        )

    subFC_mats.append(df_fc.to_numpy(dtype=float))

# Shape: subjects x selected ROIs x selected ROIs
subFC_mats = np.stack(subFC_mats, axis=0)

# ------------------------------------------------------------------
# Group-average FC across subjects
# ------------------------------------------------------------------
mean_FC = np.mean(subFC_mats, axis=0)
subFC = mean_FC.copy()

# Use the saved CSV ROI ordering as the canonical order downstream.
rois_incl = sub_rois

# Cortical parcels contain 'SomMot'; all remaining saved entries are spinal ROIs.
sm_idx = [i for i, roi in enumerate(rois_incl) if "SomMot" in roi]
sc_idx = [i for i, roi in enumerate(rois_incl) if "SomMot" not in roi]

# ROI map used by later cells, retaining hemisphere for cortical ROIs.
rois_map = [
    "LH_SomMot" if "LH_SomMot" in roi
    else "RH_SomMot" if "RH_SomMot" in roi
    else roi.split(" ")[0]
    for roi in rois_incl
]

# ------------------------------------------------------------------
# Extract and sparsify the cortical SMC-SMC block
# ------------------------------------------------------------------
cortical_FC = subFC[np.ix_(sm_idx, sm_idx)]

sparsity_cortical = 0.9
cortical_sparse = np.array([
    row * (row > np.sort(row)[int(sparsity_cortical * len(row)) - 1])
    for row in cortical_FC
])

# Insert the sparsified cortical block into the full SMC-spinal FC matrix.
FC_cs = subFC.copy()
FC_cs[np.ix_(sm_idx, sm_idx)] = cortical_sparse

#### Visualize spinal FC 

In [ ]:
# cortical_FC is your square connectivity matrix
# cortical_FC = ...
spinal_FC = subFC_spinal = subFC[62:, 62:]
# 1) Z-score and clip to [-1, 1]
spinal_z = (spinal_FC - spinal_FC.mean()) / spinal_FC.std()
spinal_z = np.clip(spinal_z, -1, 1)

# 2) CSS-like diverging colormap
colors = ["#00a2ff", "#9ddff5", "#ffffff", "#ffbfdf", "#ff369b"]
cmap_css = LinearSegmentedColormap.from_list("css_fc_div", colors, N=256)

# 3) Figure size so each cell is ~square
n = spinal_z.shape[0]
cell_size = 0.1  # inches per cell
fig_size = n * cell_size

fig, ax = plt.subplots(figsize=(fig_size, fig_size), dpi=300)

sns.heatmap(
    spinal_z,
    cmap=cmap_css,
    square=True,          # enforce square cells
    cbar=True,
    linewidths=0.25,       # thin borders
    linecolor='white',
    xticklabels=False,
    yticklabels=False,
    ax=ax
)
cbar = ax.collections[0].colorbar
cbar.set_label("Z-scored FC", rotation=270, labelpad=15)
plt.tight_layout()
plt.show()
out_path = os.path.join(
    params["save_main_sc"],
    "figure4_FC_SC_33ROI.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")

#### Reference axes for Procrustes alignment
Axis 1: Ascending WM & Dorsal horns (-1) --> Intermediate Zone (0) --> Descending WM & Ventral horns (1). Axis 2: Right GM (-1) --> Ascending & descending WM (0) --> Left GM (1)

In [ ]:
# select the group of RoIs on which to project the gradients to - i.e. all spinal RoIs at level C6
rois_spinal_x = [roi for j,roi in enumerate(rois_incl) if rois_map[j]=='C6']
# for corticospinal gradients, the y dimension of FC includes both C6 RoIs as well as SomMot RoIs
rois_corticospinal_y = [roi for j,roi in enumerate(rois_incl) if rois_map[j]=='C6'] + [roi for j,roi in enumerate(rois_incl) if 'SomMot' in roi]

# Reference axes for alignment with Procrustes
# Axis 1: Ascending WM & Dorsal horns (-1) --> Intermediate Zone (0) --> Descending WM & Ventral horns (1)
# Axis 2: Right GM (-1) --> Ascending & descending WM (0) --> Left GM (1)
ref_mat_custom = np.zeros((len(rois_spinal_x),2))
for j,roi in enumerate(rois_spinal_x):
    if roi.split(' ')[1] == 'WM':
        if roi.split(' ')[2] == 'descending':
            ref_mat_custom[j,0] = 1
        else:
            ref_mat_custom[j,0] = -1
    else:
        if roi.split(' ')[3] == 'ventral':
            ref_mat_custom[j,0] = 1
        elif roi.split(' ')[3] == 'dorsal':
            ref_mat_custom[j,0] = -1
        if roi.split(' ')[2] == 'left':
            ref_mat_custom[j,1] = 1
        else:
            ref_mat_custom[j,1] = -1           

#### Computing Gradients (spinal and corticospinal) + Alignment

The saved gradients maps are visualized with a specialized spinal map rendering on the local system. The results are presented in Panel 4A.

In [ ]:
### [spinal]
grads_spinal, lambdas_spinal, FC_spinal = fit_gradients(FC_cs, rois_spinal_x, rois_incl,
        n_components = 6, approach = 'dm', kernel = 'cosine', sparsity = 0)
ref_norm, grads_spinal_aligned, disparity_sc = align_procrustes_matlab(grads_spinal, ref_mat_custom, align_dims=2)

grads_spinal_aligned = (grads_spinal_aligned - np.mean(grads_spinal_aligned, axis=0))/np.std(grads_spinal_aligned, axis=0)

#_, grads_spinal_img = project_4D(grads_spinal_aligned, rois_spinal_x, ['C6']*len(rois_spinal_x), params)
#grads_spinal_img.to_filename(params['save_grads_sc'] + 'CS_spinal_33_C6_resting_aligned.nii.gz')

### [spinal-smc]
grads_corticospinal, lambdas_corticospinal, FC_corticospinal = fit_gradients(FC_cs, rois_spinal_x, rois_incl, rois_corticospinal_y, 
        n_components = 6, approach = 'dm', kernel = 'cosine', sparsity = 0)

ref_norm, grads_corticospinal_aligned, disparity_sc_cs = align_procrustes_matlab(grads_corticospinal, ref_mat_custom, align_dims=2)
#grads_corticospinal_aligned = (grads_corticospinal_aligned - np.mean(grads_corticospinal_aligned, axis=0))/np.std(grads_corticospinal_aligned, axis=0)

#_ , grads_corticospinal_img = project_4D(grads_corticospinal_aligned, rois_spinal_x, ['C6']*len(rois_spinal_x), params)
#grads_corticospinal_img.to_filename(params['save_grads_sc'] + 'CS_corticospinal_33_C6_resting_aligned.nii.gz')


### Print explained variance & metrics (unchanged)
print("Explained variance - Spinal:")
print(lambdas_spinal)
print("\nExplained variance - Spinal-SMC:")
print(lambdas_corticospinal)
print(f"\nDisparity spinal aligned: {disparity_sc:.3f}")
print(f"Disparity spinal+cs aligned: {disparity_sc_cs:.3f}")
print(f"\nSpinal corr(G1,G2): {np.corrcoef(grads_spinal_aligned[:,0], grads_spinal_aligned[:,1])[0,1]:.3f}")
print(f"Spinal-SMC corr(G1,G2): {np.corrcoef(grads_corticospinal_aligned[:,0], grads_corticospinal_aligned[:,1])[0,1]:.3f}")

#### Explained variance

In [ ]:
# Given (% explained variance)
x = np.arange(1, len(lambdas_spinal) + 1)
fig, ax = plt.subplots(figsize=(3, 4), dpi=300)
# Connected dots and lines
ax.plot(x, lambdas_spinal, marker='o', color='k', linewidth=1)
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance (%)')
# Shaded block along x-axis from component 1 to 2
ax.axvspan(0.95, 2.05, color='grey', alpha=0.1)
ax.set_xticks(x)
ax.set_ylim(0, max(lambdas_spinal) * 1.1)
plt.tight_layout()
plt.show()
out_path = os.path.join(
    params["save_main_sc"],
    "figure4_SC_Gradients_ExpVar_33ROI.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")

## Panel 4C (Top)

This cell computes the scatter plots for 33-ROI case (spinal-spinal gradient space).

In [ ]:
# -------------------------------------------------------------------
# Config
# -------------------------------------------------------------------

# Full 33 spinal ROIs (template without the level prefix)
rois_order_33 = [
    'GM left dorsal horn',
    'GM left intermediate zone',
    'GM left ventral horn',
    'GM right dorsal horn',
    'GM right intermediate zone',
    'GM right ventral horn',
    'WM left fasciculus cuneatus',
    'WM left fasciculus gracilis',
    'WM left lateral corticospinal tract',
    'WM left lateral vestibulospinal tract',
    'WM left medial longitudinal fasciculus',
    'WM left medial reticulospinal tract',
    'WM left rubrospinal tract',
    'WM left spinal lemniscus (spinothalamic and spinoreticular tracts)',
    'WM left spino-olivary tract',
    'WM left tectospinal tract',
    'WM left ventral corticospinal tract',
    'WM left ventral reticulospinal tract',
    'WM left ventral spinocerebellar tract',
    'WM left ventrolateral reticulospinal tract',
    'WM right fasciculus cuneatus',
    'WM right fasciculus gracilis',
    'WM right lateral corticospinal tract',
    'WM right lateral vestibulospinal tract',
    'WM right medial reticulospinal tract',
    'WM right rubrospinal tract',
    'WM right spinal lemniscus (spinothalamic and spinoreticular tracts)',
    'WM right spino-olivary tract',
    'WM right tectospinal tract',
    'WM right ventral corticospinal tract',
    'WM right ventral reticulospinal tract',
    'WM right ventral spinocerebellar tract',
    'WM right ventrolateral reticulospinal tract'
]

def clip_gradients(g, clip_val=2.0):
    """Clip gradients to [-clip_val, clip_val] along each dimension."""
    return np.clip(g, -clip_val, clip_val)


def parse_tissue_side(base_name):
    """
    base_name: e.g. 'WM right medial reticulospinal tract'
    returns: tissue ('GM'/'WM'), side ('left'/'right')
    """
    tissue = 'GM' if base_name.startswith('GM') else 'WM'
    side = 'left' if ' left ' in base_name else ('right' if ' right ' in base_name else 'unknown')
    return tissue, side


def shorten_label(base_name):
    """
    Shorten:
      'WM right medial reticulospinal tract' -> 'WM right mrt'
    i.e. keep 'WM right' + first letters of remaining words.
    """
    parts = base_name.split()
    if len(parts) <= 3:
        return base_name  # already short
    tissue, side = parts[0], parts[1]
    tail = parts[2:]
    initials = ''.join(w[0] for w in tail)
    return f"{tissue} {side} {initials}"


def make_roi_colors_per_group(base_names):
    """
    Within each group (GM, WM_left, WM_right), assign distinct colors.
    Colors can repeat across groups but not within a group.
    """
    # cmap = plt.cm.get_cmap('tab20')
    # n_colors = cmap.N
    
    cmap = mpl.colormaps["tab20"]
    n_colors = cmap.N

    gm = [r for r in base_names if r.startswith('GM')]
    wm_left = [r for r in base_names if r.startswith('WM left')]
    wm_right = [r for r in base_names if r.startswith('WM right')]

    colors = {}

    def assign(group_list, offset):
        for i, r in enumerate(group_list):
            colors[r] = cmap((offset + i) % n_colors)

    assign(gm, offset=0)
    assign(wm_left, offset=len(gm))
    assign(wm_right, offset=len(gm) + len(wm_left))

    return colors


# -------------------------------------------------------------------
# Helpers (unchanged logic except clipping)
# -------------------------------------------------------------------

def get_spinal_rois_for_level(level, rois_incl, rois_map):
    """
    level: 'C4'..'C8'
    returns:
        rois_spinal_x (list of full labels, e.g. 'C6 GM left dorsal horn')
        rois_corticospinal_y (C-level + SomMot)
    """
    rois_spinal_x = [roi for j, roi in enumerate(rois_incl) if rois_map[j] == level]
    rois_corticospinal_y = rois_spinal_x + [roi for roi in rois_incl if 'SomMot' in roi]
    return rois_spinal_x, rois_corticospinal_y


def compute_subject_gradients_for_level_33(
    FC_mat, level, rois_incl, rois_map, rois_list,
    ref_mat_custom, n_components=10, clip_val=2.0, mode='both'
):
    """
    mode: 'spinal', 'corticospinal', or 'both'
    Returns:
        grads_spinal_aligned (or None),
        grads_corticospinal_aligned (or None),
        rois_spinal_x
    """
    rois_spinal_x, rois_corticospinal_y = get_spinal_rois_for_level(level, rois_incl, rois_map)

    idx_y = [rois_list.index(roi) for roi in rois_corticospinal_y]
    FC_incl = FC_mat[np.ix_(idx_y, idx_y)]

    grads_spinal_aligned = None
    grads_corticospinal_aligned = None

    if mode in ('spinal', 'both'):
        grads_spinal, _, _ = fit_gradients(
            FC_incl, rois_spinal_x, rois_corticospinal_y,
            n_components=n_components, approach='dm', kernel='cosine', sparsity=0
        )
        _, grads_spinal_aligned, _ = align_procrustes_matlab(grads_spinal, ref_mat_custom, align_dims=2)
        #grads_spinal_aligned = grads_spinal
        grads_spinal_aligned = (grads_spinal_aligned - np.mean(grads_spinal_aligned, axis=0)) / np.std(grads_spinal_aligned, axis=0)
        grads_spinal_aligned = clip_gradients(grads_spinal_aligned, clip_val=clip_val)

    if mode in ('corticospinal', 'both'):
        grads_corticospinal, _, _ = fit_gradients(
            FC_incl, rois_spinal_x, rois_corticospinal_y, rois_corticospinal_y,
            n_components=n_components, approach='dm', kernel='cosine', sparsity=0
        )
        _, grads_corticospinal_aligned, _ = align_procrustes_matlab(grads_corticospinal, ref_mat_custom, align_dims=2)
        grads_corticospinal_aligned = (grads_corticospinal_aligned - np.mean(grads_corticospinal_aligned, axis=0)) / np.std(grads_corticospinal_aligned, axis=0)
        grads_corticospinal_aligned = clip_gradients(grads_corticospinal_aligned, clip_val=clip_val)

    return grads_spinal_aligned, grads_corticospinal_aligned, rois_spinal_x


def compute_centroids(points_per_roi):
    """points_per_roi: dict[roi_name] -> list of (N_i x 2) points from all subjects."""
    centroids = {}
    for roi, pts in points_per_roi.items():
        pts_arr = np.vstack(pts)
        centroids[roi] = pts_arr.mean(axis=0)
    return centroids

# -------------------------------------------------------------------
# Main subject-wise plotting function for 33 parcels
# -------------------------------------------------------------------
subject_list = [f'S{i:02d}' for i in range(1, 21) if i != 15]
def plot_subject_gradients_with_centroids_33(FC_mats, level,
                                          rois_incl, rois_map, rois_list,
                                          ref_mat_custom,
                                          save_dir, prefix='C_level',
                                          clip_val=None,
                                          mode='both',
                                          max_subjects=None,
                                          subject_ids=None,
                                          select_subject=None):
    """
    ...
    subject_ids: optional list of IDs matching FC_mats (e.g. ['S01', 'S02', ...])
    select_subject: optional subject ID to plot only that subject (e.g. 'S04')

    Returns:
        grads_spinal_all_33, grads_corticospinal_all_33
        shapes:
          - (n_subj_used, 33, 2) for each mode that is active
          - None if that mode not requested
    """

    base_names = rois_order_33
    roi_colors_33 = make_roi_colors_per_group(base_names)

    # Optionally select a specific subject OR restrict number of subjects
    if select_subject is not None and subject_ids is not None:
        try:
            idx = subject_ids.index(select_subject)
        except ValueError:
            raise ValueError(f"Subject {select_subject} not found in subject_ids")
        FC_mats_use = [FC_mats[idx]]
    else:
        if max_subjects is not None:
            FC_mats_use = FC_mats[:max_subjects]
        else:
            FC_mats_use = FC_mats

    n_subj_used = len(FC_mats_use)

    # Containers: per ROI (base name), stack points
    spinal_points = {roi: [] for roi in base_names}
    corticospinal_points = {roi: [] for roi in base_names}

    # NEW: arrays to store per-subject gradients (33 x 2)
    grads_spinal_all_33 = np.zeros((n_subj_used, len(base_names), 2), dtype=float) \
        if mode in ('spinal', 'both') else None
    grads_corticospinal_all_33 = np.zeros((n_subj_used, len(base_names), 2), dtype=float) \
        if mode in ('corticospinal', 'both') else None

    # Loop over subjects
    for s, FC_mat in enumerate(FC_mats_use):
        g_spinal, g_cs, rois_spinal_x = compute_subject_gradients_for_level_33(
            FC_mat, level, rois_incl, rois_map, rois_list, ref_mat_custom,
            n_components=10, clip_val=clip_val, mode=mode
        )
        # rois_spinal_x are full names for this level; map to base_names order
        current_base = [full.split(' ', 1)[1] for full in rois_spinal_x]

        for i, roi_base in enumerate(base_names):
            if roi_base not in current_base:
                continue
            idx = current_base.index(roi_base)

            if mode in ('spinal', 'both') and g_spinal is not None:
                pt_sp = g_spinal[idx, :]
                spinal_points[roi_base].append(pt_sp)
                grads_spinal_all_33[s, i, :] = pt_sp

            if mode in ('corticospinal', 'both') and g_cs is not None:
                pt_cs = g_cs[idx, :]
                corticospinal_points[roi_base].append(pt_cs)
                grads_corticospinal_all_33[s, i, :] = pt_cs

    # Compute centroids only for the modes requested
    spinal_centroids = compute_centroids(spinal_points) if mode in ('spinal', 'both') else {}
    corticospinal_centroids = compute_centroids(corticospinal_points) if mode in ('corticospinal', 'both') else {}

    # -------- plotting code unchanged below --------
    plt.figure(figsize=(8, 8), dpi = 300)

    for roi_base in base_names:
        color = roi_colors_33[roi_base]

        if mode in ('spinal', 'both') and len(spinal_points[roi_base]) > 0:
            pts_sp = np.vstack(spinal_points[roi_base])
            plt.scatter(
                pts_sp[:, 1], pts_sp[:, 0],
                s=150, alpha=0.4,
                facecolor=color, edgecolor='black', linewidth=1.0
            )

        if mode in ('corticospinal', 'both') and len(corticospinal_points[roi_base]) > 0:
            pts_cs = np.vstack(corticospinal_points[roi_base])
            plt.scatter(
                pts_cs[:, 1], pts_cs[:, 0],
                s=150, alpha=0.4,
                facecolor='none', edgecolor=color, linewidth=1.0
            )

    centroid_handles = []
    centroid_labels = []

    for roi_base in base_names:
        color = roi_colors_33[roi_base]
        tissue, side = parse_tissue_side(roi_base)

        if tissue == 'GM':
            marker = 'o'
        else:
            if side == 'left':
                marker = '^'
            elif side == 'right':
                marker = 's'
            else:
                marker = 'X'

        if mode in ('spinal', 'both') and roi_base in spinal_centroids:
            cx_sp, cy_sp = spinal_centroids[roi_base][1], spinal_centroids[roi_base][0]
            h = plt.scatter(
                cx_sp, cy_sp,
                s=300, facecolor=color, edgecolor='black', linewidth=1.5,
                marker=marker
            )
            centroid_handles.append(h)
            centroid_labels.append(shorten_label(roi_base))

        if mode in ('corticospinal', 'both') and roi_base in corticospinal_centroids:
            cx_cs, cy_cs = corticospinal_centroids[roi_base][1], corticospinal_centroids[roi_base][0]
            plt.scatter(
                cx_cs, cy_cs,
                s=300, facecolor='white', edgecolor=color, linewidth=1.5,
                marker=marker
            )

    plt.xlabel('G2', fontsize=12, fontweight='bold')
    plt.ylabel('G1', fontsize=12, fontweight='bold')
    plt.title(
        f'{level} spinal gradients across subjects (33 ROIs)\n'
        f'Mode: {mode}, clipped to ±{clip_val}',
        fontsize=14, fontweight='bold'
    )

    if centroid_handles:
        plt.legend(
            centroid_handles, centroid_labels,
            title='Centroids: tissue/side/tract',
            loc='upper center',
            bbox_to_anchor=(0.5, -0.1),
            ncol=6,
            frameon=True,
            fontsize=7
        )

    plt.tight_layout()
    plt.savefig(
        f'{save_dir}/{prefix}_{level}_scatter_centroids_33_clipped_mode-{mode}.png',
        dpi=300, bbox_inches='tight'
    )
    plt.show()

    return grads_spinal_all_33, grads_corticospinal_all_33


##### Group Level (all subjects)

In [ ]:
g33_spinal, g33_cs = plot_subject_gradients_with_centroids_33(
    FC_mats=subFC_mats,
    level='C6',
    rois_incl=rois_incl,
    rois_map=rois_map,
    rois_list=sub_rois,
    ref_mat_custom=ref_mat_custom,
    save_dir=params['save_grads_sc_scatter'],
    prefix='figure4_spinal_gradients_group',
    clip_val=2.5,
    mode='spinal',
    subject_ids=subject_list,
    select_subject=None
)

##### Single Subject

In [ ]:
g33_spinal, g33_cs = plot_subject_gradients_with_centroids_33(
    FC_mats=subFC_mats,
    level='C6',
    rois_incl=rois_incl,
    rois_map=rois_map,
    rois_list=sub_rois,
    ref_mat_custom=ref_mat_custom,
    save_dir=params['save_grads_sc_scatter'],
    prefix='figure4_spinal_gradients_single',
    clip_val=2.5,
    mode='spinal',
    subject_ids=subject_list,
    select_subject='S05'
)

## Panel 4C (Bottom) 

This cell computes the Fisher’s J separation score and GM compactness score for group and single subject level for both 33-ROI and 8-ROI case. 

#### Fisher’s J separation score (Group and Single subject)

In [ ]:
# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
base = params['save_main_sc']  # adjust if needed

# subject index for single-subject plot (0-based)
single_subj = 6

# colors (you can change these hex codes)
parc_hex_map = {
    '33 ROIs': '#f4d444',   # light gray
    '8 ROIs':  '#0b3866',   # steel blue
}

# ------------------------------------------------------------------
# Load saved gradients
# ------------------------------------------------------------------
g8  = np.load(base + '_npy/C6_spinal_gradients_8ROIs.npy')    # (n_subj, 8, 2)
g33 = np.load(base + '_npy/C6_spinal_gradients_33ROIs.npy')   # (n_subj, 33, 2)

# ROI names (must match saving order)
rois_8 = [
    'GM left dorsal horn',
    'GM left intermediate zone',
    'GM left ventral horn',
    'WM descending',
    'GM right ventral horn',
    'GM right intermediate zone',
    'GM right dorsal horn',
    'WM ascending'
]

rois_33 = [
    'GM left dorsal horn',
    'GM left intermediate zone',
    'GM left ventral horn',
    'GM right dorsal horn',
    'GM right intermediate zone',
    'GM right ventral horn',
    'WM left fasciculus cuneatus',
    'WM left fasciculus gracilis',
    'WM left lateral corticospinal tract',
    'WM left lateral vestibulospinal tract',
    'WM left medial longitudinal fasciculus',
    'WM left medial reticulospinal tract',
    'WM left rubrospinal tract',
    'WM left spinal lemniscus (spinothalamic and spinoreticular tracts)',
    'WM left spino-olivary tract',
    'WM left tectospinal tract',
    'WM left ventral corticospinal tract',
    'WM left ventral reticulospinal tract',
    'WM left ventral spinocerebellar tract',
    'WM left ventrolateral reticulospinal tract',
    'WM right fasciculus cuneatus',
    'WM right fasciculus gracilis',
    'WM right lateral corticospinal tract',
    'WM right lateral vestibulospinal tract',
    'WM right medial reticulospinal tract',
    'WM right rubrospinal tract',
    'WM right spinal lemniscus (spinothalamic and spinoreticular tracts)',
    'WM right spino-olivary tract',
    'WM right tectospinal tract',
    'WM right ventral corticospinal tract',
    'WM right ventral reticulospinal tract',
    'WM right ventral spinocerebellar tract',
    'WM right ventrolateral reticulospinal tract'
]

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def parse_tissue(name):
    name = str(name).strip()
    parts = name.split()
    if parts[0] in {'C1','C2','C3','C4','C5','C6','C7','C8'}:
        parts = parts[1:]
    core = ' '.join(parts)
    if core.startswith('GM'):
        return 'GM'
    elif core.startswith('WM'):
        return 'WM'
    else:
        return 'WM'

def mean_sem(x):
    x = np.asarray(x, float)
    m = np.mean(x)
    s = np.std(x, ddof=1) / np.sqrt(len(x))
    return m, s

def fisher_J_2class(coords, labels):
    coords = np.asarray(coords)
    labels = np.asarray(labels)
    classes = np.unique(labels)
    if len(classes) != 2:
        return np.nan
    c1, c2 = classes
    X1 = coords[labels == c1]
    X2 = coords[labels == c2]
    if X1.shape[0] < 2 or X2.shape[0] < 2:
        return np.nan
    mu1 = X1.mean(axis=0)
    mu2 = X2.mean(axis=0)
    var1 = X1.var(axis=0, ddof=1).sum()
    var2 = X2.var(axis=0, ddof=1).sum()
    num = np.sum((mu1 - mu2) ** 2)
    den = var1 + var2
    if den <= 0:
        return np.nan
    return num / den

def gmwm_labels_from_rois(rois):
    labels = np.zeros(len(rois), dtype=int)  # 0=GM, 1=WM
    for i, name in enumerate(rois):
        t = parse_tissue(name)
        labels[i] = 0 if t == 'GM' else 1
    mask_valid = np.ones(len(rois), dtype=bool)
    return labels, mask_valid

def per_subject_J(G, labels, mask_valid):
    n_subj, n_roi, _ = G.shape
    J_vals = np.zeros(n_subj, float)
    idx = np.where(mask_valid)[0]
    lab = labels[idx]
    for s in range(n_subj):
        coords = G[s, idx, :]
        J_vals[s] = fisher_J_2class(coords, lab)
    return J_vals

# ------------------------------------------------------------------
# Compute GM–WM Fisher J for 8 vs 33
# ------------------------------------------------------------------
labels_gmwm_8,  mask_gmwm_8  = gmwm_labels_from_rois(rois_8)
labels_gmwm_33, mask_gmwm_33 = gmwm_labels_from_rois(rois_33)

J_GM_8  = per_subject_J(g8,  labels_gmwm_8,  mask_gmwm_8)
J_GM_33 = per_subject_J(g33, labels_gmwm_33, mask_gmwm_33)

mask = ~np.isnan(J_GM_33) & ~np.isnan(J_GM_8)
tJ_GM, pJ_GM = ttest_rel(J_GM_33[mask], J_GM_8[mask], nan_policy='omit')
mJ_GM_33, seJ_GM_33 = mean_sem(J_GM_33[mask])
mJ_GM_8,  seJ_GM_8  = mean_sem(J_GM_8[mask])

print("GM–WM Fisher J: 33ROIs mean±SE = %.3f±%.3f,  8ROIs = %.3f±%.3f,  p=%.3g"
      % (mJ_GM_33, seJ_GM_33, mJ_GM_8, seJ_GM_8, pJ_GM))

# ------------------------------------------------------------------
# Build DataFrames for plotnine
# ------------------------------------------------------------------
# Group-level DF (means + SE)
df_group = pd.DataFrame({
    'parcellation': ['33 ROIs', '8 ROIs'],
    'FJ_mean':      [mJ_GM_33, mJ_GM_8],
    'FJ_se':        [seJ_GM_33, seJ_GM_8],
})

# Single-subject DF
single_idx = single_subj
df_single = pd.DataFrame({
    'parcellation': ['33 ROIs', '8 ROIs'],
    'FJ':           [float(J_GM_33[single_idx]), float(J_GM_8[single_idx])],
})

# ------------------------------------------------------------------
# Figure 1: group-level Fisher J
# horizontal dot plot with SE
# -----------------------------------------------------------------

labels = ['33 ROIs', '8 ROIs']
y = np.array([1, 0])

means = np.array([mJ_GM_33, mJ_GM_8])
errs = np.array([seJ_GM_33 + 0.025, seJ_GM_8 + 0.025])

fig, ax = plt.subplots(1, 1, figsize=(4, 2), dpi=300)

# subject-level points
ax.scatter(
    J_GM_33, np.full_like(J_GM_33, 1, dtype=float),
    s=40, color='#f4d444', alpha=0.4, edgecolor='black', linewidth=0.5, zorder=1
)
ax.scatter(
    J_GM_8, np.full_like(J_GM_8, 0, dtype=float),
    s=40, color='#4C78A8', alpha=0.4, edgecolor='black',linewidth=0.5, zorder=1
)

ax.scatter(
    [mJ_GM_33], [1],
    s=100, color='#f4d444', alpha=0.9,edgecolor='black', linewidth=1.0, zorder=2
)
ax.scatter(
    [mJ_GM_8], [0],
    s=100, color='#4C78A8', alpha=0.9, edgecolor='black', linewidth=1.0, zorder=2
)

ax.errorbar(
    means, y,
    xerr=errs,
    fmt='none',
    ecolor='black',
    elinewidth=1.2,
    capsize=4,
    capthick=1.2,
    zorder=1
)


ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel('Fisher J (GM–WM)')
ax.set_title(f'GM–WM Fisher J\npaired t-test p = {pJ_GM:.3g}')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add breathing room around dots / error bars
xmax = np.max(np.r_[means + errs, J_GM_33, J_GM_8])
xpad = 0.06 * xmax if xmax > 0 else 0.1
ax.set_xlim(0, xmax + xpad)

# denser x ticks
xticks = np.linspace(0, 1, 6)
ax.set_xticks(xticks)
ax.set_ylim(-0.6, 1.6)

fig.subplots_adjust(left=0.30, right=0.95, top=0.83, bottom=0.22)

plt.savefig(
    base + '/figure4_C6_FisherJ_GMWM_group_dotplot_8_vs_33.png',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.2
)
plt.show()


# ------------------------------------------------------------------
# Figure 2: single-subject Fisher J
# horizontal dot plot without error bars
# ------------------------------------------------------------------
single_subj = 6  # set subject index

J33_s = float(J_GM_33[single_subj])
J8_s  = float(J_GM_8[single_subj])

labels = ['33 ROIs', '8 ROIs']
y = np.array([1, 0])
vals = np.array([J33_s, J8_s])

fig, ax = plt.subplots(1, 1, figsize=(4, 2), dpi=300)

ax.scatter(
    [J33_s], [1],
    s=140, color='#f4d444', edgecolor='black', linewidth=1.2, zorder=3
)
ax.scatter(
    [J8_s], [0],
    s=140, color='#4C78A8', edgecolor='black', linewidth=1.2, zorder=3
)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel('Fisher J (GM–WM)')
ax.set_title(f'GM–WM Fisher J\nsubject {single_subj}')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# denser x ticks
xticks = np.linspace(0, 0.2, 6)
ax.set_xticks(xticks)
ax.set_ylim(-0.6, 1.6)

fig.subplots_adjust(left=0.30, right=0.95, top=0.83, bottom=0.22)

plt.savefig(
    base + f'/figure4_C6_FisherJ_GMWM_singleSubj_{single_subj}_dotplot_8_vs_33.png',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.2
)
plt.show()

#### GM Compactness score (Group and Single subject)

In [ ]:
# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
base = params['save_main_sc']   # adjust if needed
single_subj = 6                 # subject index for single-subject plot (0-based)

# ------------------------------------------------------------------
# Load gradients
# ------------------------------------------------------------------
g8  = np.load(base + '_npy/C6_spinal_gradients_8ROIs.npy')    # (n_subj, 8, 2)
g33 = np.load(base + '_npy/C6_spinal_gradients_33ROIs.npy')   # (n_subj, 33, 2)

rois_8 = [
    'GM left dorsal horn',
    'GM left intermediate zone',
    'GM left ventral horn',
    'WM descending',
    'GM right ventral horn',
    'GM right intermediate zone',
    'GM right dorsal horn',
    'WM ascending'
]

rois_33 = [
    'GM left dorsal horn',
    'GM left intermediate zone',
    'GM left ventral horn',
    'GM right dorsal horn',
    'GM right intermediate zone',
    'GM right ventral horn',
    'WM left fasciculus cuneatus',
    'WM left fasciculus gracilis',
    'WM left lateral corticospinal tract',
    'WM left lateral vestibulospinal tract',
    'WM left medial longitudinal fasciculus',
    'WM left medial reticulospinal tract',
    'WM left rubrospinal tract',
    'WM left spinal lemniscus (spinothalamic and spinoreticular tracts)',
    'WM left spino-olivary tract',
    'WM left tectospinal tract',
    'WM left ventral corticospinal tract',
    'WM left ventral reticulospinal tract',
    'WM left ventral spinocerebellar tract',
    'WM left ventrolateral reticulospinal tract',
    'WM right fasciculus cuneatus',
    'WM right fasciculus gracilis',
    'WM right lateral corticospinal tract',
    'WM right lateral vestibulospinal tract',
    'WM right medial reticulospinal tract',
    'WM right rubrospinal tract',
    'WM right spinal lemniscus (spinothalamic and spinoreticular tracts)',
    'WM right spino-olivary tract',
    'WM right tectospinal tract',
    'WM right ventral corticospinal tract',
    'WM right ventral reticulospinal tract',
    'WM right ventral spinocerebellar tract',
    'WM right ventrolateral reticulospinal tract'
]

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def normalize_core_name(name):
    name = str(name).strip()
    parts = name.split()
    if parts[0] in {'C1','C2','C3','C4','C5','C6','C7','C8',
                    'T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12'}:
        parts = parts[1:]
    return ' '.join(parts)

def parse_tissue_from_core(core):
    if core.startswith('GM'):
        return 'GM'
    elif core.startswith('WM'):
        return 'WM'
    else:
        return 'WM'

def mean_sem(x):
    x = np.asarray(x, float)
    m = np.mean(x)
    s = np.std(x, ddof=1) / np.sqrt(len(x))
    return m, s

def gm_compactness(coords):
    """
    coords: (n_gm, 2), GM-only.
    Returns mean squared distance to GM centroid.
    """
    if coords.shape[0] < 2:
        return np.nan
    mu = coords.mean(axis=0)
    d2 = np.sum((coords - mu)**2, axis=1)
    return np.mean(d2)

def global_spread(coords_all):
    """
    coords_all: (n_roi, 2) all ROIs for a subject/parcellation.
    Spread = sqrt(trace(cov)).
    """
    if coords_all.shape[0] < 2:
        return 1.0
    cov = np.cov(coords_all.T)
    sp = np.sqrt(np.trace(cov))
    return sp if sp > 0 else 1.0

# ------------------------------------------------------------------
# Common GM ROIs (by core name)
# ------------------------------------------------------------------
rois_8_core  = [normalize_core_name(r) for r in rois_8]
rois_33_core = [normalize_core_name(r) for r in rois_33]

idx8_core_map  = {core: i for i, core in enumerate(rois_8_core)}
idx33_core_map = {core: i for i, core in enumerate(rois_33_core)}

common_gm_names = []
gm_idx_8 = []
gm_idx_33 = []

for core in rois_8_core:
    if parse_tissue_from_core(core) != 'GM':
        continue
    if core in idx33_core_map:
        common_gm_names.append(core)
        gm_idx_8.append(idx8_core_map[core])
        gm_idx_33.append(idx33_core_map[core])

gm_idx_8  = np.array(gm_idx_8, dtype=int)
gm_idx_33 = np.array(gm_idx_33, dtype=int)

print("Common GM ROIs (core names) used for compactness (normalized):")
for name, i8, i33 in zip(common_gm_names, gm_idx_8, gm_idx_33):
    print(f"  {name}: 8-ROI idx={i8}, 33-ROI idx={i33}")

if len(gm_idx_8) == 0:
    raise RuntimeError("No common GM ROIs between 8 and 33 parcellations.")

# ------------------------------------------------------------------
# Compute spread-normalized GM compactness per subject
# ------------------------------------------------------------------
n_subj = g8.shape[0]
C_GM_8  = np.zeros(n_subj, float)
C_GM_33 = np.zeros(n_subj, float)

for s in range(n_subj):
    coords8_all  = g8[s]
    coords33_all = g33[s]
    coords8_gm   = g8[s,  gm_idx_8,  :]
    coords33_gm  = g33[s, gm_idx_33, :]

    comp8  = gm_compactness(coords8_gm)
    comp33 = gm_compactness(coords33_gm)

    sp8  = global_spread(coords8_all)
    sp33 = global_spread(coords33_all)

    C_GM_8[s]  = comp8  / (sp8**2)
    C_GM_33[s] = comp33 / (sp33**2)

mask = ~np.isnan(C_GM_33) & ~np.isnan(C_GM_8)
tC, pC = ttest_rel(C_GM_33[mask], C_GM_8[mask], nan_policy='omit')
mC_33, seC_33 = mean_sem(C_GM_33[mask])
mC_8,  seC_8  = mean_sem(C_GM_8[mask])

print("Spread-normalized GM compactness (mean sq dist / spread^2):")
print("  33ROIs mean±SE = %.3f±%.3f,  8ROIs = %.3f±%.3f,  p=%.3g"
      % (mC_33, seC_33, mC_8, seC_8, pC))

# ------------------------------------------------------------------
# Plot 1: group-level horizontal dot plot with subject-level points
# ------------------------------------------------------------------
labels = ['33 ROIs', '8 ROIs']
y = np.array([1, 0])

means = np.array([mC_33, mC_8])
errs  = np.array([seC_33 + 0.025, seC_8 + 0.025])

fig, ax = plt.subplots(1, 1, figsize=(4, 2), dpi=300)

# subject-level points
ax.scatter(
    C_GM_33, np.full_like(C_GM_33, 1, dtype=float),
    s=40, color='#f4d444', alpha=0.4, edgecolor='black', linewidth=0.5, zorder=1
)
ax.scatter(
    C_GM_8, np.full_like(C_GM_8, 0, dtype=float),
    s=40, color='#4C78A8', alpha=0.4, edgecolor='black', linewidth=0.5, zorder=1
)

# mean points
ax.scatter(
    [mC_33], [1],
    s=100, color='#f4d444', alpha=0.9, edgecolor='black', linewidth=1.0, zorder=2
)
ax.scatter(
    [mC_8], [0],
    s=100, color='#4C78A8', alpha=0.9, edgecolor='black', linewidth=1.0, zorder=2
)

# SE bars
ax.errorbar(
    means, y,
    xerr=errs,
    fmt='none',
    ecolor='black',
    elinewidth=1.2,
    capsize=4,
    capthick=1.2,
    zorder=1
)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel('Normalized GM compactness')
ax.set_title(f'GM compactness\npaired t-test p = {pC:.3g}')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

xmax = np.max(np.r_[means + errs, C_GM_33, C_GM_8])
xpad = 0.06 * xmax if xmax > 0 else 0.1
ax.set_xlim(0, xmax + xpad)

xticks = np.linspace(0, xmax + xpad, 6)
ax.set_xticks(xticks)
ax.set_ylim(-0.6, 1.6)

fig.subplots_adjust(left=0.30, right=0.95, top=0.83, bottom=0.22)

plt.savefig(
    base + '/figure4_C6_GM_compactness_norm_group_dotplot_8_vs_33.png',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.2
)
plt.show()

# ------------------------------------------------------------------
# Plot 2: single-subject horizontal dot plot (no SE)
# ------------------------------------------------------------------
C33_s = float(C_GM_33[single_subj])
C8_s  = float(C_GM_8[single_subj])

vals = np.array([C33_s, C8_s])

fig, ax = plt.subplots(1, 1, figsize=(4, 2), dpi=300)

ax.scatter(
    [C33_s], [1],
    s=100, color='#f4d444', alpha=0.9, edgecolor='black', linewidth=1.0, zorder=2
)
ax.scatter(
    [C8_s], [0],
    s=100, color='#4C78A8', alpha=0.9, edgecolor='black', linewidth=1.0, zorder=2
)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel('Normalized GM compactness')
ax.set_title(f'GM compactness\nsubject {single_subj}')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

xmax = np.max(vals)
xpad = 0.06 * xmax if xmax > 0 else 0.1
ax.set_xlim(0, xmax + xpad)

xticks = np.linspace(0, xmax + xpad, 6)
ax.set_xticks(xticks)
ax.set_ylim(-0.6, 1.6)

fig.subplots_adjust(left=0.30, right=0.95, top=0.83, bottom=0.22)

plt.savefig(
    base + f'/figure4_C6_GM_compactness_norm_singleSubj_{single_subj}_dotplot_8_vs_33.png',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.2
)
plt.show()